# 强化学习与大模型后训练 · 第 5/12 课：PPO：比率裁剪与训练循环

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现 clipped surrogate，并解释 old policy、epoch 与 on-policy 边界。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、概率期望、基本深度学习
- 本课在路线中的作用：PPO 用新旧策略概率比率衡量更新幅度，并裁剪会让目标继续变好的过大比率。

## 核心心智模型

### 1. 它是什么，解决什么问题

PPO 用新旧策略概率比率衡量更新幅度，并裁剪会让目标继续变好的过大比率。

### 2. 它如何工作

ratio=exp(logπ_new-logπ_old)；对正优势限制比率上界，对负优势限制下界，最终取两个 surrogate 的最小值。

### 3. 正确性条件与常见误区

old log-prob 必须冻结且与样本同源；重复 epoch 太多会让数据越来越 off-policy，clip 不能提供硬 KL 保证。

### 4. 性能与工程取舍

更小 clip/更少 epoch 稳定但学习慢；可同时监控 approximate KL、clip fraction 和 entropy。

## 具体演示

A>0、ratio=1.4、ε=.2 时贡献被截到 1.2A；A<0 时 `min` 的方向常被写错。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐标量 PPO clipped surrogate。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import math

def ppo_objective(logp_new, logp_old, advantage, eps):
    ratio = math.exp(logp_new - logp_old)
    clipped = min(max(ratio, 1.0 - eps), 1.0 + eps)
    # TODO：PPO 最大化两个 surrogate 中更保守的那个。
    return ______

assert abs(ppo_objective(math.log(1.4), 0.0, 2.0, .2) - 2.4) < 1e-9
assert abs(ppo_objective(math.log(.6), 0.0, -2.0, .2) - (-1.6)) < 1e-9


### 检查方法

运行正、负 advantage 两个断言；再画出 ratio∈[0.5,1.5] 的目标折线。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“PPO：比率裁剪与训练循环”的工作机制。

**你的答案：**


### Q2

为什么“ratio 被 clip”不等于“策略 KL 一定小”？

**你的答案：**


### Q3

若 clip fraction 接近 0 但 reward 不升，应优先查哪些信号？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
import math

def ppo_objective(logp_new, logp_old, advantage, eps):
    ratio = math.exp(logp_new - logp_old)
    clipped = min(max(ratio, 1.0 - eps), 1.0 + eps)
    return min(ratio * advantage, clipped * advantage)

assert abs(ppo_objective(math.log(1.4), 0.0, 2.0, .2) - 2.4) < 1e-9
assert abs(ppo_objective(math.log(.6), 0.0, -2.0, .2) - (-1.6)) < 1e-9


### Q1 参考答案

ratio=exp(logπ_new-logπ_old)；对正优势限制比率上界，对负优势限制下界，最终取两个 surrogate 的最小值。

### Q2 参考答案

判断时先检查本课不变量：old log-prob 必须冻结且与样本同源；重复 epoch 太多会让数据越来越 off-policy，clip 不能提供硬 KL 保证。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：更小 clip/更少 epoch 稳定但学习慢；可同时监控 approximate KL、clip fraction 和 entropy。

## 参考资料

- [PPO paper](https://arxiv.org/abs/1707.06347)

资料用于建立事实基线；面试回答仍需用自己的语言组织。